# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset package—"Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya"—using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
This notebook accesses the dataset via a Croissant schema at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

All dataset resources (record sets, fields, etc.) are referenced by their Croissant `@id` identifiers.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and explore core details using the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Metadata object, not a dict

print(f"Dataset Title: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', 'N/A')}")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', 'N/A')}")

print("\nDataset Description:")
print(metadata.description)

print("\nData Collection Approach:")
print(getattr(metadata, 'dataCollection', 'N/A'))

## 2. Data Overview
Explore available record sets in the Croissant dataset, with all references by their `@id` fields.

We first enumerate the `@id` values for each record set, along with their associated fields and columns.

In [ ]:
# List all available record sets by `@id`, and describe their fields/columns
print("Available record sets:`@id`")

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rset in record_sets:
        print(f"\n- Record Set @id: {rset['@id']}")
        print(f"  Name: {rset.get('name', 'N/A')}")
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields/Columns:")
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id')
                field_name = field.get('name', 'N/A')
            else:
                field_id = field
                field_name = 'N/A'
            print(f"    - {field_id} (name: {field_name})")

## 3. Data Extraction
For each available record set, we load the records into a pandas DataFrame using their Croissant record set `@id`.

**Note:** All references must use the corresponding `@id`, which you can confirm from the previous Data Overview output.

In [ ]:
# If there are record sets, load each into a pandas DataFrame by `@id`
dataframes = {}
record_set_ids = [rset['@id'] for rset in dataset.record_sets]

if not record_set_ids:
    print("[Warning] No record sets detected, so no tabular data to extract.")
else:
    for record_set_id in record_set_ids:
        print(f"\nLoading records for record set: {record_set_id}")
        # Each record is a dict (field_id -> value)
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame with columns: {list(df.columns)} (Rows: {len(df)})")
        else:
            print(f"[No records returned for record set: {record_set_id}]")

# For demonstration, pick the first available record set if any:
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nPreview for record set @id: {first_id}")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
We will:
- Select a numeric field (`@id`) from one record set.
- Filter records above a threshold.
- Normalize the results for that field.
- Group the data by a categorical (non-numeric) field if available.

Update the `numeric_field_id` and `group_field_id` below based on your own dataset fields (see the code outputs above for the correct field `@id`s).

In [ ]:
# EXAMPLE: Replace these IDs based on actual data overview output

if not dataframes:
    print("No record sets loaded, skipping EDA.")
else:
    # Pick the first loaded DataFrame for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Operating on record set: {record_set_id}")

    # Guess possible numeric fields
    numeric_field_candidates = df.select_dtypes(include=[float, int]).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Numeric field candidate: {numeric_field_id}")
    else:
        numeric_field_id = df.columns[0]
        print("No numeric columns found. Using the first column as a placeholder.")

    threshold = 10
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
    except Exception as e:
        print(f"Could not apply threshold filter: {e}")
        filtered_df = df.copy()

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    try:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Normalization failed: {e}")

    # Try to guess a categorical field
    group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
    else:
        group_field_id = None

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped mean values by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Now visualize a numeric field using basic plotting. Adjust `numeric_field_id` and `group_field_id` as needed for your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data available to visualize.")
else:
    df = dataframes[record_set_id]
    # Histogram or boxplot for numeric field
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Boxplot by group field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded and examined metadata and structure of the FAIR\^2 dataset via its Croissant schema.
- Explored available record sets and their fields using Croissant `@id` references.
- Loaded available tabular data, performed basic filtering, normalization, and grouping.
- Visualized the structure and distributions of selected fields.

**Note:**
- All references in the notebook are made using Croissant `@id` values for reproducibility and clarity.
- To perform deeper analyses, refer specifically to the field and record set `@id`s tailored to your own schema inspection.